# GG4257 - Creating Sustainable Cities with Spatial Data Science
## Lab 2 Challenges
All code cells will include a brief explanation of that code line, in cases where I have used websites or documentation, please refer to the [x] section of the 'references' cell at the end of the notebook to view citations.

### 220016532


# Challenge No 1:

1. Using a Dictionary, create a dataframe (table), with at least 4 columns and more than 100 rows. How come you can create this among data from scratch without defining every single row of data? 
2. Using the appropriate method, create a new DataFrame containing only the first 30 rows and the first 3 columns of the original DataFrame. Name this new DataFrame subset_df.
3. Using the appropriate method, filter the rows from the original dataframe where a numerical attribute(column) is greater than a particular numerical value, and find another categorical attribute that is equal to a specific string or text. Name this new DataFrame filtered_df.
4. Check this website https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.mean.html and apply the methods, mean, standard deviation, group_by to run fundamental statistical analysis of your created data frame.
5. Make sure you comment on your code and describe how you are manipulating the data.


In [1]:
# importing libraries
import numpy as np 
import pandas as pd


Task 1: 


In [182]:
years = range(2000, 2026) #creates a list of consecutive integers from 2000 to 2026
a = np.repeat(years, 4) #26*4 = 104 rows (uses code from lab)
b = np.random.randint(0, 40, 104) # generate random integers - 104 values between 0 and 26
c = np.random.choice (['A', 'B', 'C', 'D', 'E'], 104) # generates random categories from a-e for 104 rows as string is required for task 3
d = np.random.randint(0, 40, 104) # generate random integers - 104 values between 0 and 26

In [183]:
df = pd.DataFrame({'a':a,'b':b,'c':c,'d':d})  #the curly brackets indicate a dictionary

Task 2 :

In [184]:
subset_df = df.iloc[:30,:3] #using iloc to filter by positions and : to include all data in 30 rows and 3 columns
subset_df

,a,b,c
0,2000,24,B
1,2000,33,C
2,2000,3,E
3,2000,9,B
4,2001,19,A
5,2001,39,D
6,2001,24,D
7,2001,3,D
8,2002,39,A
9,2002,17,D


Task 3:

In [185]:
filtered_df = df[(df['b']>10) & (df['c']=='B')] #takes b as it is numerical and filters for values >10, '&' also requires column c (string) to be equal (==) to B
filtered_df

,a,b,c,d
0,2000,24,B,2
29,2007,39,B,25
39,2009,32,B,22
40,2010,39,B,28
43,2010,15,B,23
49,2012,23,B,27
65,2016,17,B,35
67,2016,11,B,5
73,2018,26,B,17
77,2019,38,B,33


Task 4:

In [186]:
df_mean = df.mean(numeric_only=True) #had to add numberic_only=True as dataframe includes strings and integers

df_std = df.std(numeric_only=True)

df_groupby= df.groupby(["c"]).mean(numeric_only=True) # groups dataframe by values in column c, and averages all columns in that group.

print(df_mean, df_std, df_groupby) # prints all methods

a    2012.500000
b      20.557692
d      19.230769
dtype: float64 a     7.536320
b    11.195376
d    12.227798
dtype: float64              a          b          d
c                                   
A  2014.318182  21.454545  19.909091
B  2012.133333  21.400000  20.133333
C  2012.823529  24.470588  20.235294
D  2011.208333  19.166667  19.416667
E  2012.153846  18.038462  17.307692


# Challenge No 2:

**Part No 1:**

1. Using the same workflow previously described, now calculate the clustered areas for the GeoPandasDataFrame `gdf_bikes_end`
2. Make sure you don't have any NaN in your columns, add a CRS, clean up the unnecessary attributes, calculate the cluster values, and plot a map of 4 calculated clusters for the return locations.


In [187]:
import requests
import geopandas as gpd

# Part 1 :

Note : gdf_bikes_end was made as part of the lab instructions, and so I am importing that code and recreating that geodataframe here.

In [188]:
# code from lab below
url_bikes = "https://api.glasgow.gov.uk/mobility/v1/get_rentals?startDate=2022-05-01&endDate=2023-05-01" #glasgow api url
response = requests.get(url_bikes) # making the query to the web server, using the Get method from the requests library 
data = response.json() # get the response from the web server, we need to translate that into a format we can manipulate, like JSON.
rental_data = data['data'] # usually, there are two labels into the web server response the metadata, and the data; we will use the data label
rental_pd = pd.DataFrame(rental_data) # creates pandas dataframe

In [189]:
nan_in_column_Lat = rental_pd['endPlaceLat'].isna().sum() # checks for total NaNs in each column, found 33
nan_in_column_Long = rental_pd['endPlaceLong'].isna().sum()
print(nan_in_column_Lat,nan_in_column_Lat) 

33 33


In [190]:
clean_rental_pd = rental_pd.dropna(subset=['startPlaceLat', 'startPlaceLong', 'endPlaceLat','endPlaceLong']) # removes NaNs in all long/lat columns

In [191]:
gdf_bikes_end = gpd.GeoDataFrame(clean_rental_pd, geometry=gpd.points_from_xy(clean_rental_pd['endPlaceLong'], clean_rental_pd['endPlaceLat'])) # creates geodataframe, using long/lat columns to create xy points.

In [192]:
gdf_bikes_end = gdf_bikes_end.set_crs("EPSG:4326") # sets crs as epsg:4326

In [193]:
# gdf_bikes_end.explore() # checking this worked, remove # to run

In [194]:
# code from lab, changed to keep cols relevant to bikes_end instead of bikes_start
keep_cols = [ #sets columns to keep in geodataframe, as not all are needed
    "endDate",
    "endPlaceId",
    "endPlaceName",
    "durationSeconds",
    "isInvalid",
    "price",
    "isEbike",
    "endPlaceLat",
    "endPlaceLong",
    "geometry",
]
gdf_bikes_end = gdf_bikes_end[keep_cols]

gdf_bikes_end.info() # shows dtypes to change in next cell.

<class 'geopandas.geodataframe.GeoDataFrame'>
Index: 9940 entries, 0 to 9999
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype   
---  ------           --------------  -----   
 0   endDate          9940 non-null   object  
 1   endPlaceId       9940 non-null   object  
 2   endPlaceName     9940 non-null   object  
 3   durationSeconds  9940 non-null   int64   
 4   isInvalid        9940 non-null   bool    
 5   price            9940 non-null   float64 
 6   isEbike          9940 non-null   bool    
 7   endPlaceLat      9940 non-null   float64 
 8   endPlaceLong     9940 non-null   float64 
 9   geometry         9940 non-null   geometry
dtypes: bool(2), float64(3), geometry(1), int64(1), object(3)
memory usage: 718.3+ KB


In [195]:
gdf_bikes_end.endPlaceId = gdf_bikes_end.endPlaceId.astype(int)
gdf_bikes_end.endPlaceName = gdf_bikes_end.endPlaceName.astype(str)
gdf_bikes_end['endDate'] = pd.to_datetime(gdf_bikes_end['endDate'], format='%Y-%m-%dT%H:%M:%SZ')

In [196]:
# gdf_bikes_end.dtypes # checking no more dtypes need to be changed

### Clustering step :

In [217]:
from sklearn.cluster import KMeans
# code edited from lab
num_clusters = 4
kmeans_collection = KMeans(n_clusters=num_clusters, random_state=42)
gdf_bikes_end['kmeans_cluster'] = kmeans_collection.fit_predict(gdf_bikes_end[['endPlaceLong', 'endPlaceLat']]) # fit_predict from sklearn requires lat and long columns

In [218]:
gdf_bikes_end.head() # verify kmeans_cluster column has been created

,endDate,endPlaceId,endPlaceName,durationSeconds,isInvalid,price,isEbike,endPlaceLat,endPlaceLong,geometry,kmeans_cluster
0,2022-05-01 00:02:36,264299,ELECTRIC - Broomielaw,56,False,2.0,True,55.856600,-4.263521,POINT (-4.26352 55.85660),3
1,2022-05-01 00:22:34,3812776,Alexandra Park (south entrance) Alexandra Para...,1097,False,1.0,False,55.863128,-4.210282,POINT (-4.21028 55.86313),0
2,2022-05-01 00:10:08,264302,Botanic Gardens - ELECTRIC,344,False,1.0,False,55.878278,-4.288487,POINT (-4.28849 55.87828),1
3,2022-05-01 02:57:49,349455,ELECTRIC - Cessnock Subway Station,10400,False,6.0,False,55.851918,-4.294490,POINT (-4.29449 55.85192),1
4,2022-05-01 01:21:31,264295,ELECTRIC - St Enoch Square,4611,False,3.0,False,55.856829,-4.255292,POINT (-4.25529 55.85683),3


Note : leafmap is not part of the ua.yml file, so it must be run once in the cell below.

In [219]:
# !pip install leafmap

In [220]:
import leafmap

In [221]:
# again, code imported from lab.

m = leafmap.Map(center=(55.860166, -4.257505), #sets default zoom to Glasgow
                zoom=12,
                draw_control=False,
                measure_control=False,
                fullscreen_control=False,
                attribution_control=True,
                   
               )

m.add_basemap("OpenStreetMap") # I had to change basemap here, I selected OpenStreetMap from the provided list
m.add_data(
    gdf_bikes_end,
    column='kmeans_cluster', # uses previously created column
    legend_title='Clusters',
    cmap='Set1', 
    k=4,
)

#plotting map
m

OpenStreetMap has been already added before.


Map(center=[55.860166, -4.257505], controls=(AttributionControl(options=['position', 'prefix'], position='bott…

**Part No 2:**

1. Using the Glasglow Open Data API ( Transit) https://developer.glasgow.gov.uk/api-details#api=traffic&operation=traffic-sensor-locations fetch all the sensor locations in the city.
2. Map the sensor
3. Find the WorkingZones and Calculate/Map the areas with more and fewer sensors distributed in the city.
4. You will need:
   * Get two separate Geopandas DataFrames, one for the traffic sensors and another one for the WorkingZones.
   * Using `sJoin` ( Spatial Join) https://geopandas.org/en/stable/docs/reference/api/geopandas.sjoin.html
   calculate the overlay of sensors and polygons.
   * Using group_by https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.groupby.html to count the number of sensors per WorkingZone
   * Make sure you add the counts into the WorkingZone polygons of Glasgow so you can create a map of Zones with more and fewer traffic sensors.
   * Of course, you will need extra steps where you manipulate the data and extract what you need, for instance, clipping the Working Zones only for Glasgow.
5. Make sure you comment on your code and describe how you are manipulating the data.


In [202]:
sensors = "http://api.glasgow.gov.uk/traffic/v1/movement/sites"
# uses Get method, same as previous code
response = requests.get(sensors)
# response
data = response.json() # translates to json
# data # in markdown after running cell as output is very long

In [203]:
# code from lab
# empty lists to store what we need
site_ids = []
from_lats = []
from_longs = [] 
from_descriptions = []

# Iterate over each entry in the JSON array
for entry in data:
    site_id = entry['siteId']
    from_description = entry['from']['description']
    from_lat = entry['from']['lat']
    from_long = entry['from']['long']
    
    # Append values to lists
    site_ids.append(site_id)
    from_descriptions.append(from_description)
    from_lats.append(from_lat)
    from_longs.append(from_long)
    
# Create a Pandas DataFrame, using the lists.
df = pd.DataFrame({
    'siteId': site_ids,
    'from_description': from_descriptions,
    'from_lat': from_lats,
    'from_long': from_longs,
})

In [204]:
df.head() # viewing dataframe columns made in previous cell

,siteId,from_description,from_lat,from_long
0,GH020A_B,Gallowgate west to Sword St,55.8543754827825,-4.217259197086495
1,GJ3301_D,Carmunnock Rd southbound,55.821736212748405,-4.2561270616272875
2,GH020A_A,Gallowgate east to Sword St,55.85539015389372,-4.225769520669405
3,GJ3301_C,Carmunnock Rd northbound to Kings Park Avenue,55.81925458152306,-4.255696036412074
4,GJ3301_B,Kings Park Ave westbound to Carmunnock Rd,55.820288451422314,-4.250775113480551


In [205]:
gdf_sensors = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df['from_long'], df['from_lat']), crs='EPSG:4326') # converts to geodataframe for mapping , using long/lat columns and setting crs

### Sensors map :

In [206]:
m = leafmap.Map(center=(55.860166, -4.257505), # reuses map settings from previous cell / lab
                zoom=12,
                draw_control=False,
                measure_control=False,
                fullscreen_control=False,
                attribution_control=True,
                   
               )
m.add_basemap("OpenStreetMap") 
m.add_gdf(gdf_sensors, layer_name="siteId") # simpler map adding geodataframe taken from 'reading spatial data' section of lab
m

OpenStreetMap has been already added before.


Map(center=[55.860166, -4.257505], controls=(AttributionControl(options=['position', 'prefix'], position='bott…

### Workplace zones map :

In [207]:
# code taken from lab
m = leafmap.Map(
    center=(56.329031,-3.798943),
    zoom=7
)
wms_url = 'https://maps.gov.scot/server/services/NRS/Census2011/MapServer/WMSServer?'
# A WMS URL include multiple layers, so you need to provide the name you need to load in your map.
# See this: https://www.spatialdata.gov.scot/geonetwork/srv/eng/catalog.search#/metadata/ff882746-e913-4f78-862e-f6e3974fb80e


m.add_wms_layer(url=wms_url, layers='WorkplaceZones2011', name='Census2011', shown=True) # adds specifically the WorkplaceZones2011 layer, this can be changed to something else (e.g OutputArea2011) by looking at the metadata linked above.
# m

Upon rendering this map, I realised I need to subset the data to Glasgow only. For this, i cannot use the map layer from the WMS url, and I instead use the shapefile provided in the lab's data folder to subset the data and add a spatial join later.

In [208]:
zones = gpd.read_file('data/WorkplaceZones2011Scotland/WorkplaceZones2011Scotland.shp')
zones.head()

,LADCD,WZCD,geometry
0,S12000040,S34005373,"POLYGON ((304339.365 666492.616, 304341.959 66..."
1,S12000033,S34000001,"POLYGON ((393511.570 805880.709, 393506.643 80..."
2,S12000033,S34000002,"POLYGON ((393557.291 806046.379, 393563.505 80..."
3,S12000033,S34000003,"POLYGON ((394178.803 806251.348, 394195.604 80..."
4,S12000033,S34000004,"POLYGON ((394179.722 806307.033, 394181.625 80..."


From these columns, I realise that each polygon is grouped into a local authority zone (ladcd), and checking online i was able to find Glasgow's code, S12000046 (there was some confusion with this, I had to make sure I was using the LADCD code from 2011, as this later changed to S12000049.

In [209]:
# code edited from 'subsetting data section of lab
glasgow_zones = zones.loc[zones["LADCD"] == 'S12000046', :] # subsets to only include rows where the LADCD code column is Glasgow's local authority code
glasgow_zones.head()
glasgow_zones.explore() # quickly checks if this is accurate

In [210]:
glasgow_zones.crs # the crs is EPSG : 27700, so i must convert it to EPSG: 4326, i had to use to_crs instead of set_crs to change this. [1]
glasgow_zones = glasgow_zones.to_crs(4326)
glasgow_zones.crs

<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

Spatial join :

In [211]:
zones_sensors_joined = gpd.sjoin(gdf_sensors, glasgow_zones, how='inner') # edited spatial join code from lab, which joins the two geodataframes
zones_sensors_joined.explore()

In [212]:
zones_sensors_joined = zones_sensors_joined[["geometry","siteId","WZCD"]] # now i can remove unnecessary columns. i keep the 'wzcd' column as this is a unique code for each polygon in my shapefile.
zones_sensors_joined.head()

,geometry,siteId,WZCD
0,POINT (-4.21726 55.85438),GH020A_B,S34004861
1,POINT (-4.25613 55.82174),GJ3301_D,S34004978
2,POINT (-4.22577 55.85539),GH020A_A,S34005204
3,POINT (-4.25570 55.81925),GJ3301_C,S34004966
4,POINT (-4.25078 55.82029),GJ3301_B,S34004838


In [213]:
zones_sensors_count = zones_sensors_joined.groupby("WZCD").size() # uses groupby and .size to count how many rows (points) each WZCD has [2]
zones_sensors_count = zones_sensors_count.reset_index(name="sensor_count")# here i realised i was making a series before using reset_index, i found a fix for this online to convert this into a dataframe and name the column to merge back with my glasgow_zones geodataframe [2]
zones_sensors_count.head()

,WZCD,sensor_count
0,S34000363,1
1,S34000365,1
2,S34000367,5
3,S34000368,2
4,S34000370,5


In [214]:
glasgow_zones = glasgow_zones.merge(zones_sensors_count, on="WZCD", how="left") # merges my created 'count' column onto the old dataframe, as i was able to keep the 'WZCD' column and use on=

# when using .explore(), i realised that i had to add how='left', as the merge only kept WZCD rows that had atleast one sensor, meaning that the polygon would not render on my map [3].I can also see this when i .head() zones_sensor_count in the cell above, as some WZCD rows are missing
# by default, .merge sets how='inner', which only keeps matching WZCD rows between dataframes. how='left' ensures that all rows on the left side of the merge (glasgow_zones) are kept

glasgow_zones

,LADCD,WZCD,geometry,sensor_count
0,S12000046,S34000364,"POLYGON ((-4.28071 55.86787, -4.28037 55.86819...",NaN
1,S12000046,S34000365,"POLYGON ((-4.27158 55.86770, -4.27154 55.86767...",1.0
2,S12000046,S34000366,"POLYGON ((-4.27858 55.86645, -4.27854 55.86646...",NaN
3,S12000046,S34000367,"POLYGON ((-4.25858 55.85750, -4.25902 55.85751...",5.0
4,S12000046,S34000368,"POLYGON ((-4.26175 55.86113, -4.26181 55.86090...",2.0
...,...,...,...,...
634,S12000046,S34005147,"POLYGON ((-4.26185 55.87438, -4.26282 55.87406...",4.0
635,S12000046,S34000408,"POLYGON ((-4.25989 55.86186, -4.25929 55.86178...",6.0
636,S12000046,S34004876,"POLYGON ((-4.32097 55.87624, -4.32089 55.87621...",NaN
637,S12000046,S34000410,"MULTIPOLYGON (((-4.25916 55.85923, -4.25920 55...",NaN


In [215]:
glasgow_zones["sensor_count"].isna().sum() # here i count the polygons or zones without any sensors.
glasgow_zones = glasgow_zones.fillna(0) # i still wanted these zones to show on the map, so i was able to use pandas to just replace these with 0 [4]

In [216]:
glasgow_zones.explore(column="sensor_count", cmap="OrRd") # i had completed lab 3 before this challenge, so just displayed this as a choropleth the same way as in lab 3.

## References
- [1] https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoDataFrame.to_crs.html#
- [2] https://stackoverflow.com/questions/19384532/get-statistics-for-each-group-such-as-count-mean-etc-using-pandas-groupby
- [3] https://pandas.pydata.org/docs/reference/api/pandas.merge.html
- [4] https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.fillna.html

### Extra notes :
I also found an alternative fix for my series/dataframe problem when using .groupby() by using .to_frame() instead of reset_index(), I thought I'd include this here as it may be useful to look back on.
- https://bobbyhadz.com/blog/value-error-cannot-merge-series-without-name